# **4. Métodologia Modelos**

## **4.1. Pipeline Metodológico General**

Mediante el flujograma se describe la estructura metodológica común que comparten los modelos benchmark, estableciendo un marco uniforme que garantiza su comparabilidad.

![Pipeline](Imagenes/pipeline_metodologia_general.png)

**1. Carga de datos (Ingestión)**
Se leen archivos CSV principales, se parsean fechas y se ordenan temporalmente por departamento. Los inputs son `df_full.csv` y `centroides_departamentos.csv`, usando pandas, pathlib y os.

**2. Feature Engineering (Transformación)**
Se generan 29 features y 1 target mediante codificación cíclica de tiempo (seno/coseno), variables epidemiológicas con rezagos de 11 variables, 18 variables climáticas (temperatura, humedad, precipitación) y transformación logarítmica de la incidencia como target.

**3. Folds Espaciales (Partición espacial)**
Se crean 5 particiones geográficas mediante centroides UTM con `estimate_utm_crs()`, clustering K-Means geográfico (EPSG:4326 → UTM) y mapeo de folds al dataframe principal usando `spatialkfold` y `geopandas`.

**4. Esquema de Validación (Estrategia CV)**
Se implementa Nested Temporal Cross-Validation con outer test en 2022/23/24, inner folds temporales en 2019/2020/2021, Spatial-KFold interno de 5 pliegues × 3 temporal y LeaveOneGroupOut por fold retenido. Se ejecutan 15 iteraciones por trial de Optuna.

**5. Ventanas y Escalamiento (Preprocesamiento)**
Se usan ventanas deslizantes con `create_windows()`, tensores de forma (B, T, N, F) pivoteados por departamento, `RobustScaler` ajustado solo en train y DataLoaders con `STTDataset` y pin_memory. El window_size y horizon se fijan en 4.

**6. Arquitectura del Modelo**
Especificada por modelo, contempla encoders de entrada, módulos de atención, fusión multimodal y capa de predicción, con parámetros `d_model`, `n_heads` y `n_layers`, implementados en `torch.nn` / PyTorch.

**7. Optimización de Hiperparámetros (HPO)**
Se usa Optuna con TPESampler (20 trials/seed), función objetivo de RMSE promedio CV, poda de trials inválidos con TrialPruned, múltiples seeds [42, 123, 2024] y espacio de búsqueda sobre lr, batch, dropout, window, d_model y max_lag, con EarlyStopping.

**8. Evaluación e Interpretabilidad**
Se reportan métricas globales (RMSE, MAE, NSE, KGE, sMAPE), RMSE_top10 y SkillScore, Pearson_r y Bias, análisis de residuos (ACF, PACF, Ljung-Box) y mapas de interpretabilidad con SHAP + Attention maps vía GradientExplainer. Los outputs guardados incluyen predictions, residuals, losses, shap y visualizaciones con statsmodels y matplotlib.

## **4.2. Métricas Usadas**

### **4.2.1. Métrica Principal**

El **Root Mean Square Error (RMSE)** se define como la raíz cuadrada del promedio de los errores al cuadrado entre los valores observados y los valores predichos:

\[
$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}$
\]

Esta métrica expresa la magnitud típica del error en las mismas unidades de la variable analizada. Su principal característica es que penaliza de manera más fuerte los errores grandes, lo que la hace especialmente útil cuando se desea evaluar la precisión de un modelo bajo supuestos de normalidad en la distribución de los errores.

El **Root Mean Square Error (RMSE)** fue adoptado como la métrica principal para la evaluación del desempeño predictivo debido a sus propiedades teóricas y a su pertinencia para el problema de predicción de incidencia de dengue. Esta métrica cuantifica la magnitud promedio de los errores de predicción en las mismas unidades de la variable analizada y, al elevar las diferencias al cuadrado antes de promediarlas, asigna una penalización mayor a los errores de gran magnitud.

Esta característica resulta especialmente relevante en el contexto epidemiológico, donde las desviaciones importantes entre los valores observados y los predichos pueden tener consecuencias significativas para la planificación y la toma de decisiones en salud pública. En particular, la subestimación de periodos de alta incidencia o de eventos epidémicos puede afectar la asignación de recursos, la capacidad de respuesta hospitalaria y la implementación oportuna de estrategias de vigilancia y control.

Adicionalmente, el RMSE mantiene coherencia con el proceso de optimización empleado durante el entrenamiento de los modelos de deep learning. Dado que la mayoría de estos modelos son entrenados mediante la minimización del Error Cuadrático Medio (MSE), el RMSE constituye una medida de evaluación directamente relacionada con la función objetivo utilizada durante el aprendizaje, favoreciendo la consistencia entre la fase de entrenamiento y la fase de validación.

Finalmente, considerando la marcada heterogeneidad espacial de la incidencia de dengue entre los departamentos de Colombia, la evaluación sobre la escala logarítmica de la variable objetivo permite reducir el efecto de las diferencias extremas en magnitud entre regiones de alta y baja endemicidad. En este contexto, el RMSE calculado sobre los valores transformados proporciona una medida más equilibrada del desempeño global, facilitando comparaciones robustas entre departamentos con patrones epidemiológicos significativamente distintos.


### **4.2.2. Otras Métricas Usadas**

| Métrica | Definición | Fórmula | Interpretación | Relevancia epidemiológica |
|---|---|---|---|---|
| **MAE** | Error absoluto medio. Promedia las desviaciones absolutas entre predicción y valor real, sin penalización adicional a los errores grandes. |$ (1/n) Σ \|ŷᵢ − yᵢ\|$ | Valores menores son mejores. Más robusto que el RMSE ante outliers. Interpretación directa en unidades de la variable. | Indica el error promedio en número de casos. Útil para comunicar la exactitud del modelo a tomadores de decisión en salud pública, al ser de fácil interpretación clínica. |
| **MAPE** | Error porcentual absoluto medio. Expresa el error como porcentaje del valor real observado. | $(100/n) Σ \|ŷᵢ − yᵢ\| / \|yᵢ\|$ | Valores menores son mejores. Indefinido cuando yᵢ = 0. Sesgado hacia la sobreestimación del error en valores bajos. | Permite comparar el error relativo entre departamentos con cargas endémicas muy distintas. Problemático en períodos sin casos reportados. |
| **sMAPE** | MAPE simétrico. Corrige el sesgo del MAPE usando el promedio del valor real y predicho en el denominador. | $(200/n) Σ \|ŷᵢ − yᵢ\| / (\|yᵢ\| + \|ŷᵢ\|)$ | Rango [0%, 200%]. Valores menores son mejores. Más estable que el MAPE cuando los valores reales son cercanos a cero. | Adecuado para enfermedades con temporadas de baja transmisión. Trata simétricamente la sobreestimación y la subestimación. |
| **NSE** | Eficiencia de Nash-Sutcliffe. Compara el error del modelo contra el error de usar la media observada como predictor de referencia. |$1 − [ Σ(ŷᵢ − yᵢ)² / Σ(yᵢ − ȳ)² ]$ | Rango (−∞, 1]. NSE = 1: perfecto. NSE = 0: igual que la media. NSE < 0: peor que la media. | Evalúa si el modelo captura la dinámica temporal de la transmisión y reproduce la forma de los ciclos epidémicos. |
| **KGE** | Eficiencia de Kling-Gupta. Descompone el error en tres componentes: correlación, sesgo relativo y variabilidad relativa. |$ 1 − √[ (r−1)² + (α−1)² + (β−1)² ]$ donde α = σŷ/σy, β = μŷ/μy | Rango (−∞, 1]. Valor óptimo = 1. Penaliza de forma balanceada errores en correlación, bias y varianza. | Diagnóstico multidimensional. Identifica si las fallas provienen de subestimar picos, de sesgo sistemático o de desfase temporal en la curva epidémica. |
| **Skill Score** | Ganancia relativa de un modelo sobre un predictor de referencia (media histórica u otro benchmark). | SS = 1 − RMSEmodelo / RMSEref | SS = 1: perfecto. SS = 0: igual al benchmark. SS < 0: peor que el benchmark. | Mide la utilidad operativa del modelo frente a la práctica epidemiológica habitual. Un SS positivo justifica su adopción en sistemas de vigilancia y alerta temprana. |
| **RMSE top10** | RMSE calculado exclusivamente sobre el 10% de semanas o departamentos con mayor incidencia observada. | $√( (1/k) Σᵢ∈top10% (ŷᵢ − yᵢ)² )$ | Misma escala que el RMSE global. Valores menores indican mejor desempeño en los períodos de mayor carga. | La respuesta sanitaria se activa durante los picos. Un RMSE top10 bajo indica que el modelo es confiable precisamente cuando más se necesita. |
| **Pearson r** | Coeficiente de correlación lineal entre los valores predichos y observados. |$ r = Σ(ŷᵢ − μŷ)(yᵢ − μy) / (n σŷ σy)$ | Rango [−1, 1]. r = 1: correlación perfecta. r = 0: sin asociación lineal. No captura bias ni escala absoluta. | Mide si el modelo reproduce el patrón temporal de la epidemia. Una r alta con RMSE elevado indica que el modelo sigue la tendencia pero con sesgo sistemático. |
| **Bias** | Error sistemático medio. Diferencia promedio entre predicciones y valores observados, con signo. |$ (1/n) Σ (ŷᵢ − yᵢ)$ | Bias = 0: sin sesgo. Bias > 0: sobreestimación. Bias < 0: subestimación. No captura la magnitud del error absoluto. | Un sesgo sostenido puede saturar innecesariamente los sistemas de respuesta o llevar a subreacciones ante brotes reales. Esencial para la calibración del modelo. |

> **Notación:** ŷ = valor predicho · y = valor observado · ȳ = media observada · n = número de observaciones · r = correlación · α = razón de desviaciones estándar · β = razón de medias